In [1]:
import numpy as np

In [ ]:
# Input(2) → Hidden(8) → Hidden(6) → Hidden(4) → Output(1)

#   x1, x2
#      ↓
#   a0 = [x1, x2]
#      ↓
#   z^(1) = [z1, z2, z3, z4, z5, z6, z7, z8]
#      ↓
#   a^(1) = ReLU(z^(1))
#      ↓
#   z^(2) = [z1, z2, z3, z4, z5, z6]
#      ↓
#   a^(2) = ReLU(z^(2))
#      ↓
#   z^(3) = [z1, z2, z3, z4]
#      ↓
#   a^(3) = ReLU(z^(3))
#      ↓
#   z^(4) = [z1]
#      ↓
#   a^(4) = sigmoid(z^(4))
#      ↓
#   output = 0 or 1

In [22]:
class DeepNeuralNetwork:
    
    def __init__(self, layer_sizes, learning_rate):

        self.lr = learning_rate
        self.layer_sizes = layer_sizes               # list of layer sizes
        self.num_layers = len(layer_sizes) - 1       # Weight matrices, bias vectors

        self.weights = []
        self.biases = []

        for i in range(self.num_layers):
            # Weights and biases intialization
            w = np.random.randn(layer_sizes[i], layer_sizes[i+1]) * np.sqrt(2 / layer_sizes[i])
            b = np.zeros((1, layer_sizes[i+1]))

            self.weights.append(w)
            self.biases.append(b)


    def relu(self, z):
        return np.maximum(0, z)  
    
    def relu_derivative(self, z):
        return (z > 0).astype(float)
    
    def sigmoid(self, z):
        return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

     
    def forward(self, X):
        self.activations = [X]
        self.z_values = []

        a = X
        for i in range(self.num_layers):    
            z = np.dot(a, self.weights[i]) + self.biases[i]
            self.z_values.append(z)
            
            if i == self.num_layers - 1:
                a = self.sigmoid(z)
            else:
                a = self.relu(z)
            self.activations.append(a)

        return a

    
    def backward(self, X, y, y_hat):
        m = X.shape[0]         

        delta = y_hat - y                 #dL/dz =  dL/da . da/dz,  derivative of(Sigmoid + BCE)
        for i in reversed(range(self.num_layers)):             
            a_prev = self.activations[i]              

            dW = (1/m) * np.dot(a_prev.T, delta)                      
            db = (1/m) * np.sum(delta, axis=0, keepdims=True)  # dL/db = dL/dz . 1    

            self.weights[i] -= self.lr * dW 
            self.biases[i] -= self.lr * db

            if i > 0:
                delta = np.dot(delta, self.weights[i].T) * self.relu_derivative(self.z_values[i-1])


    def train(self, X, y, epochs, print_every=50):
        for epoch in range(epochs):
            y_hat = self.forward(X)
            self.backward(X, y, y_hat) 
       

            if epoch % print_every == 0:
                loss = np.mean((y_hat - y) ** 2)
                print(f"Epoch {epoch:5d} | Loss: {loss: 6f}")


    def predict(self, X):
        output = self.forward(X)
        return (output > 0.5).astype(int)

In [ ]:
# XOR
X = np.array([[0, 0], 
              [0, 1],
              [1, 0],
              [1, 1]])

y = np.array([[0],
              [1],
              [1],
              [0]])

nn = DeepNeuralNetwork(layer_sizes=[2, 8, 6, 4, 1], learning_rate=0.1)

nn.train(X, y, epochs=1000)

Epoch     0 | Loss:  0.256926
Epoch    50 | Loss:  0.245582
Epoch   100 | Loss:  0.232922
Epoch   150 | Loss:  0.185164
Epoch   200 | Loss:  0.078919
Epoch   250 | Loss:  0.032279
Epoch   300 | Loss:  0.015708
Epoch   350 | Loss:  0.008753
Epoch   400 | Loss:  0.005460
Epoch   450 | Loss:  0.003676
Epoch   500 | Loss:  0.002624
Epoch   550 | Loss:  0.001959
Epoch   600 | Loss:  0.001512
Epoch   650 | Loss:  0.001200
Epoch   700 | Loss:  0.000974
Epoch   750 | Loss:  0.000805
Epoch   800 | Loss:  0.000677
Epoch   850 | Loss:  0.000575
Epoch   900 | Loss:  0.000495
Epoch   950 | Loss:  0.000431


In [21]:
print(nn.predict(X))

[[0]
 [1]
 [1]
 [0]]
